# 2.2 内存结构：数据的楼层分布图

上一节看到数据要“旅行”四级存储，本节把这四层楼一次讲透：**每层多大、谁住得起、谁能访问谁**。这是第二块积木，也是 Tiling 设计（第3篇）的直接依据。

## 1. 为什么要了解存储层级

第2章的瓶颈分析已经给出答案，这里用一句话复述：

> **片下内存（GM）容量大但带宽低，片上存储（L1/L0/UB）带宽高但容量小**——输入动辄几百 MB，片上只有几十 KB 到 MB 级。放不下，就只能“切块”分批处理——这就是 Tiling 的物质根源。

950 的内存层级全景（自下而上越来越快、越来越小）：

<img src="./images/950_memory_hierarchy.png" width="600">

> **看图要点**：从片外 HBM（GB 级）到 L2（128MB）、再到 AI 子系统共享的 L1、最后进入核内的 L0A/L0B/L0C 与 UB。**越靠近计算单元，容量越小、带宽越高、地位越“私有”**。

## 2. 各级存储清单

| 层级 | 位置 | 容量量级 | 归属 | FA 中的角色 |
|--|--|--|--|--|
| **GM（Global Memory）** | 片外 HBM | GB 级 | 全部核共享 | Q/K/V 输入、O 输出、KV Cache |
| **L2 Cache** | 片内 Die 级 | ~128MB（950） | Die 内共享 | 跨核数据复用 |
| **L1 Buffer** | 子系统级 | ~1MB 级 | 1 AIC + 2 AIV 共享 | Q/K/V 分块的中转站 |
| **L0A / L0B** | Cube 私有 | ~64KB | Cube 输入缓存 | 分形后的矩阵 A / B |
| **L0C** | Cube 私有 | ~256KB | Cube 输出缓存 | S 矩阵、O 累加结果 |
| **UB（Unified Buffer）** | Vector 私有 | ~256KB 级 | Vector 工作空间 | S/P 中间量、Online Softmax 全过程 |
| **BT / FP Buffer** | AIC 私有 | 4KB 级 | 分离架构新增小仓库 | Bias（BT）/ 量化与 ReLU 参数（FP） |

> **容量是 Tiling 的预算表**：第3篇推导 FA 的 Tiling 参数（单次处理的 Q 块、KV 块尺寸）时，本质就是一道“L1/UB 容量约束下的规划题”。数字因芯片型号而异，记住量级即可。

> **三个补充细节**：
> 1. **BT/FP Buffer**：分离架构给 AIC 新增的两个 KB 级小仓库（BiasTable 存 Bias、Fixpipe Parameter 存量化/ReLU 参数），由 MTE1 从 L1 搬入——第6章量化 FA 的缩放系数就住在 FP Buffer；
> 2. **Scalar 的粮仓**：Scalar 有自己的 ICache/DCache（各 16~32KB），分别缓存代码段与数据段——Kernel 代码应尽量精简，避免指令反复从 GM 取，这也是调优建议“减少 if/else 与标量运算”的原因之一；
> 3. **容量在涨**：UB 从 910B 的 192KB 涨到 910D/950 的 256KB 级，L0C 从 128KB 涨到 256KB——Tiling 的预算表逐代放宽（详见 1.2 节三代对比表）。

## 3. 计算单元的“私有领地”

存储层级最重要的规则只有一条：

> **计算单元只能直接访问自己的 Local Memory**——Cube 只能读 L0A/L0B、写 L0C；Vector 只能在 UB 上计算。任何数据想被计算，必须先被**搬运**到位。

```text
GM ──MTE2──▶ L1 ──MTE1──▶ L0A/L0B ──mmad──▶ L0C ──MTE3/FIXPIPE──▶ UB / GM
 │                                                          │
 └──────────── MTE2 直达 UB（Vector 算子走这条）◀────────────┘
```

这条规则解释了两件事：
1. 为什么必须有专职**搬运指令**（下一节 2.3 的主角）；
2. 为什么 FA 要精心编排搬运顺序——数据到不了位，再强的 Cube 也只能空转（流水线优化，第7~8章）。

## 4. 数据排布格式：ND 与分形

存储层级之外，还有一个同样重要的维度：**数据在内存里怎么摆**。

### 4.1 行优先与列优先

<img src="./images/format_row_major.png" width="440">

<img src="./images/format_col_major.png" width="440">

> **看图要点**：同一个二维矩阵，行优先（Row Major）按行连续存放，列优先（Column Major）按列连续存放。多维数组在内存中终究是一维的，排布方式决定了“相邻元素是谁”，进而决定访存效率。

### 4.2 Cube 的分形格式（NZ）

Cube 的脉动阵列一次消费 16×16 的数据块，因此它希望内存里的矩阵**天然按 16×16 小块组织**——这就是分形（Fractal）格式：

<img src="./images/format_nz.svg" width="560">

> **看图要点**：NZ 格式把矩阵切成 16×16 的分形小块，块内按列存放。Cube 从 L1 取数进入 L0A/L0B 时，MTE1 搬运指令会**顺带完成 ND → NZ 的转换**（反向同理）。这就是 2.1 节 MatMul 数据流中“排布转换”的确切含义。使用高阶 MatMul API 时，这些转换由 API 内部处理（第3篇直接受益），但理解底层机制才能在调优时看懂 Profiler 数据。

## 小结

| 关键词 | 一句话 |
|--|--|
| 层级结构 | GM → L2 → L1 → L0A/B/C / UB，越深越快越小 |
| 私有领地 | Cube 只碰 L0，Vector 只碰 UB——搬运是宿命 |
| 容量即预算 | Tiling 参数 = L1/UB 容量约束下的规划题 |
| 排布格式 | ND（常规）/ NZ（Cube 分形），搬运时自动转换 |
| 小仓库 | BT/FP Buffer 与 ICache/DCache，KB 级但各司其职 |

下一节 [2.3 搬运指令](01.05_data_move_instructions.ipynb)：认识专职“搬运工”——MTE 家族。

## 课后练习

1.（单选）下列哪个存储是 Vector Core 的工作空间？（A. L0A　B. L0C　C. UB　D. L1）

2.（判断）Cube 可以直接读取 L1 Buffer 中的数据进行矩阵乘计算。

3.（单选）ND → NZ 的分形排布转换通常发生在哪条搬运通路上？（A. GM → L1　B. L1 → L0A/L0B　C. L0C → GM　D. UB → GM）

> 完成后查看 [answer/01.04_answer.txt](answer/01.04_answer.txt) 核对答案。